# Investigating numpy-cupy compatibility

In [4]:
import sys; sys.path.insert(0,'../..')
import cupy as cp
import numpy as np


In [23]:
x=cp.arange(4).reshape((2,2)).astype(np.float32)
y = np.ones_like(x,shape=(3,3),dtype=np.int32)
y = np.zeros_like(x,shape=(3,3))
y = np.full_like(x,3.,shape=(3,3))

In [24]:
x,y

(array([[0., 1.],
        [2., 3.]], dtype=float32),
 array([[3., 3., 3.],
        [3., 3., 3.],
        [3., 3., 3.]], dtype=float32))

In [25]:
x.flat[2],x.dtype,y.dtype

(array(2., dtype=float32), dtype('float32'), dtype('float32'))

In [29]:
np.max(x.get(),initial=4.)

4.0

In [35]:
x[1]=np.nan
cp.nanmean(x)

array(0.5, dtype=float32)

In [33]:
b = x>x/2+1
np.packbits(b,bitorder='big')

TypeError: packbits() got an unexpected keyword argument 'bitorder'

In [40]:
a = cp.array([[10, 30, 20], [60, 40, 50]])
np.sort(a, axis=1); print(a)
ai = np.argsort(a, axis=1); ai
np.take_along_axis(a, ai, axis=1)

[[10 30 20]
 [60 40 50]]


array([[10, 20, 30],
       [40, 50, 60]])

In [44]:
a = cp.array([[10, 30, 20], [60, 40, 50]])
ai = np.expand_dims(np.argmax(a, axis=1), axis=1)
np.put_along_axis(a, ai, 99, axis=1)
a

TypeError: no implementation found for 'numpy.put_along_axis' on types that implement __array_function__: [<class 'cupy.core.core.ndarray'>]

In [51]:
arr = np.array([[3,6,6],[4,5,1]])
np.ravel_multi_index(arr, (7,6)),np.ravel_multi_index(arr, (7,6), order='F'),np.ravel_multi_index(arr, (4,6), mode='clip'),np.ravel_multi_index(arr, (4,4), mode=('clip','wrap'))

(array([22, 41, 37], dtype=int64),
 array([31, 41, 13], dtype=int64),
 array([22, 23, 19], dtype=int64),
 array([12, 13, 13], dtype=int64))

In [6]:
import cupyx.scipy.sparse.linalg.lsqr as solver
#cupy.cupyx

ModuleNotFoundError: No module named 'cupyx.scipy.sparse.linalg.lsqr'

In [5]:
cupyx.scipy.sparse.linalg.lsqr

<function cupyx.scipy.sparse.linalg.solve.lsqr(A, b)>

In [1]:
import cupy as cp
import numpy as np
import sys; sys.path.insert(0,"../..")

In [16]:
np.zeros(shape=(0,))

array([], dtype=float64)

In [18]:
cp.arange(10,dtype=np.int32).dtype

dtype('int32')

In [14]:
spmod = cp.cupyx.scipy.sparse

data=cp.array([1.,2.],dtype=np.float32)
row = cp.array([0,1],dtype=np.int32)
col = cp.array([0,1],dtype=np.int32)
spmat = spmod.coo_matrix((data,(row,col)))
print(spmat.dtype)
print(spmat.todense())
print(spmat.tocsr())

spmod.linalg.lsqr(spmat,cp.array([2.,3.],dtype=np.float32))

float32
[[1. 0.]
 [0. 2.]]
  (0, 0)	1.0
  (1, 1)	2.0


(array([2. , 1.5]), None, None, None, None, None, None, None, None, None)

In [2]:
from agd import Metrics 
from agd import LinearParallel as lp

In [3]:
metric = Metrics.Riemann(cp.eye(2,dtype=np.float32))
metric2 = metric.rescale((0.5,2.))

In [4]:
metric2.m

array([[4.  , 0.  ],
       [0.  , 0.25]], dtype=float32)

In [5]:
lp.trace(metric2.m)

array(4.25, dtype=float32)

In [10]:
sum(metric2.m[i,i] for i in range(2))

array(4.25, dtype=float32)

In [6]:
metric2.cost_bound()

array(2.0615528, dtype=float32)

In [3]:
(cp.array(1.,dtype=np.float64)*cp.array(2.,dtype=np.float32)).dtype

dtype('float64')

In [8]:
np.mean(cp.eye(2))

array(0.5)

In [7]:
np.broadcast_to(cp.array(1.),3)

TypeError: object of type 'int' has no len()

In [2]:
from agd import AutomaticDifferentiation as ad

In [5]:
type(ad.array([[1,2],[3,4]]))

numpy.ndarray

In [3]:
zz = cp.zeros((2,2))

In [4]:
np.array(zz)

ValueError: object __array__ method not producing an array

## Familiar funcs

In [6]:
zz2 = np.reshape(zz,(4,))
type(zz2)

In [10]:
zz2 = np.stack((zz,zz),axis=0)
type(zz2)

cupy.core.core.ndarray

In [19]:
zz2 = np.broadcast_to(zz,(2,2,2))
type(zz2)

cupy.core.core.ndarray

## No left operand bug 

In [16]:
z = cp.array(0.)
type(np.float(1.)+z)

cupy.core.core.ndarray